In [ ]:
# !pip install tensorflow_hub

In [2]:
# %pip install setuptools

Note: you may need to restart the kernel to use updated packages.


In [1]:
import tensorflow_hub as hub

ModuleNotFoundError: No module named 'pkg_resources'

In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import numpy as np

# 1. 미리 학습된 모델 로드 (인터넷 연결 필요)
model = hub.load("https://tfhub.dev/google/openimages_v4/ssd/mobilenet_v2/1").signatures['default']

# 2. 영상 불러오기
cap = cv2.VideoCapture('cat.mp4')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 모델 입력을 위해 이미지 전처리 (0~255 -> 0~1)
    input_tensor = tf.convert_to_tensor(frame, dtype=tf.float32)[tf.newaxis, ...] / 255.0
    
    # 3. 사물 인식 실행!
    results = model(input_tensor)

    # 결과값 추출 (좌표, 라벨, 확률)
    boxes = results['detection_boxes'].numpy()
    labels = results['detection_class_entities'].numpy()
    scores = results['detection_scores'].numpy()

    # 4. 화면에 그리기 (확률 0.3 이상인 것만)
    h, w, _ = frame.shape
    for i in range(len(scores)):
        if scores[i] > 0.3:
            ymin, xmin, ymax, xmax = boxes[0][i]
            # 좌표 변환 (0~1 비율 -> 픽셀 좌표)
            left, right, top, bottom = int(xmin * w), int(xmax * w), int(ymin * h), int(ymax * h)
            
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
            cv2.putText(frame, labels[i].decode('ascii'), (left, top-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow('Object Detection', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'pkg_resources'